In [18]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
import cv2
import os
import pathlib

# Set Hadamard matrix order (must divide 512 evenly)
n = 8  # Can be 2, 4, 8, 16, 32, 64, 128, 256, or 512

if 512 % n != 0:
    raise ValueError(f"512 must be divisible by n={n}. Choose n such that 512 % n == 0.")

# Generate Hadamard matrix
hadamard = sp.linalg.hadamard(n)

# Split into binary masks
h_pos = (hadamard + 1) / 2  # Convert to binary mask (0 or 1)
h_neg = np.abs((hadamard - 1) / 2)

# Compute scaling factor (each Hadamard element becomes block_size x block_size)
block_size = 512 // n

# Final canvas resolution
mask_res = 512
canvas_w, canvas_h = 1024, 768

# Output folder setup
downloads_path = str(pathlib.Path.home() / "Downloads")
output_dir = os.path.join(downloads_path, "test H masks")
os.makedirs(output_dir, exist_ok=True)

def upscale_mask(mask_2d):
    """Upscale n x n mask to 512 x 512 using block replication"""
    upscaled = np.repeat(np.repeat(mask_2d, block_size, axis=0), block_size, axis=1) * 255
    return upscaled.astype(np.uint8)

def embed_in_canvas(mask):
    """Embed 512 x 512 mask in center of 1024 x 768 canvas"""
    canvas = np.zeros((canvas_h, canvas_w), dtype=np.uint8)
    x_offset = (canvas_w - mask_res) // 2
    y_offset = (canvas_h - mask_res) // 2
    canvas[y_offset:y_offset+mask_res, x_offset:x_offset+mask_res] = mask
    return canvas

# Generate and save all vertical masks
for i in range(n):
    # Create vertical pattern by transposing direction of Hadamard vector
    pos_mask_2d = np.outer(np.ones(n), h_pos[i])
    neg_mask_2d = np.outer(np.ones(n), h_neg[i])

    # Upscale and embed
    pos_mask = upscale_mask(pos_mask_2d)
    neg_mask = upscale_mask(neg_mask_2d)

    pos_canvas = embed_in_canvas(pos_mask)
    neg_canvas = embed_in_canvas(neg_mask)

    # Save to PNG
    cv2.imwrite(os.path.join(output_dir, f'rank_{n}_pos_mask_{i}.bmp'), pos_canvas)
    cv2.imwrite(os.path.join(output_dir, f'rank_{n}_neg_mask_{i}.bmp'), neg_canvas)

print(f"Saved {n} vertical positive and {n} vertical negative masks in '{output_dir}' as 1024x768 images.")


Saved 8 vertical positive and 8 vertical negative masks in 'C:\Users\adity\Downloads\test H masks' as 1024x768 images.


In [19]:
import numpy as np
import cv2
import os
import pathlib

# --- INPUT: Set n here ---
n = 8  # You can change this or wrap in input("Enter n: ")

# Check compatibility
if 512 % n != 0:
    raise ValueError(f"512 must be divisible by n={n}. Try 2, 4, 8, 16, 32, 64, 128, 256, or 512.")

block_size = 512 // n
canvas_w, canvas_h = 1024, 768
mask_res = 512

# Output folder
downloads_path = str(pathlib.Path.home() / "Downloads")
output_dir = os.path.join(downloads_path, f"slit_masks_n{n}")
os.makedirs(output_dir, exist_ok=True)

def upscale_mask(mask_2d):
    """Upscale mask to 512x512 by repeating blocks"""
    upscaled = np.repeat(np.repeat(mask_2d, block_size, axis=0), block_size, axis=1) * 255
    return upscaled.astype(np.uint8)

def embed_in_canvas(mask):
    """Embed 512x512 in center of 1024x768 canvas"""
    canvas = np.zeros((canvas_h, canvas_w), dtype=np.uint8)
    x_offset = (canvas_w - mask_res) // 2
    y_offset = (canvas_h - mask_res) // 2
    canvas[y_offset:y_offset+mask_res, x_offset:x_offset+mask_res] = mask
    return canvas

# --- Generate slit masks ---
for i in range(n):
    # Create n x n binary mask with a vertical slit in the i-th column
    slit = np.zeros((n, n))
    slit[:, i] = 1  # Vertical bar at column i

    upscaled_slit = upscale_mask(slit)
    final_image = embed_in_canvas(upscaled_slit)

    path = os.path.join(output_dir, f"slit_mask_{i}.bmp")
    cv2.imwrite(path, final_image)

print(f"Saved {n} slit masks in '{output_dir}' with vertical bars across all {n} positions.")


Saved 8 slit masks in 'C:\Users\adity\Downloads\slit_masks_n8' with vertical bars across all 8 positions.


In [14]:
import numpy as np
import cv2
import os
import pathlib

# Settings
n = 8  # Base mask size (8x8)
if 512 % n != 0:
    raise ValueError(f"512 must be divisible by n={n}.")

block_size = 512 // n
canvas_w, canvas_h = 1024, 768
mask_res = 512

# Output directory
downloads_path = str(pathlib.Path.home() / "Downloads")
output_dir = os.path.join(downloads_path, "corner_pixel_masks")
os.makedirs(output_dir, exist_ok=True)

# Corner positions (row, col)
corners = {
    "top_left": (0, 0),
    "top_right": (0, n - 1),
    "bottom_left": (n - 1, 0),
    "bottom_right": (n - 1, n - 1),
}

# Helper functions
def upscale_mask(mask):
    return np.repeat(np.repeat(mask, block_size, axis=0), block_size, axis=1) * 255

def embed_in_canvas(mask):
    canvas = np.zeros((canvas_h, canvas_w), dtype=np.uint8)
    x_offset = (canvas_w - mask_res) // 2
    y_offset = (canvas_h - mask_res) // 2
    canvas[y_offset:y_offset+mask_res, x_offset:x_offset+mask_res] = mask.astype(np.uint8)
    return canvas

# Generate and save four corner images
for name, (r, c) in corners.items():
    mask = np.zeros((n, n), dtype=np.uint8)
    mask[r, c] = 1

    upscaled = upscale_mask(mask)
    final_image = embed_in_canvas(upscaled)

    path = os.path.join(output_dir, f"corner_{name}.jpg")
    cv2.imwrite(path, final_image)

print(f"Saved 4 corner pixel masks in '{output_dir}'.")


Saved corner mask at: C:\Users\adity\Downloads\corner_mask\four_corner_mask.png


In [15]:
import numpy as np
import cv2
import os
import pathlib

# Parameters
n = 8  # Size of the base mask
if 512 % n != 0:
    raise ValueError(f"512 must be divisible by n={n} for proper upscaling.")

block_size = 512 // n
canvas_w, canvas_h = 1024, 768
mask_res = 512

# Output path
downloads_path = str(pathlib.Path.home() / "Downloads")
output_dir = os.path.join(downloads_path, "corner_mask")
os.makedirs(output_dir, exist_ok=True)

# Create base mask
mask = np.zeros((n, n), dtype=np.uint8)
mask[0, 0] = 1           # Top-left
mask[0, -1] = 1          # Top-right
mask[-1, 0] = 1          # Bottom-left
mask[-1, -1] = 1         # Bottom-right

# Upscale to 512x512
upscaled = np.repeat(np.repeat(mask, block_size, axis=0), block_size, axis=1) * 255

# Embed in 1024x768 canvas
canvas = np.zeros((canvas_h, canvas_w), dtype=np.uint8)
x_offset = (canvas_w - mask_res) // 2
y_offset = (canvas_h - mask_res) // 2
canvas[y_offset:y_offset+mask_res, x_offset:x_offset+mask_res] = upscaled

# Save image
output_path = os.path.join(output_dir, "four_corner_mask.jpg")
cv2.imwrite(output_path, canvas)

print(f"Saved corner mask at: {output_path}")


Saved corner mask at: C:\Users\adity\Downloads\corner_mask\four_corner_mask.jpg


In [17]:
import numpy as np
import cv2
import os
import pathlib

# Image canvas size
canvas_w, canvas_h = 1024, 768

# List of white box sizes (width x height)
box_sizes = [
    (2, 2),
    (8, 8),
    (16, 16),
    (32, 32),
    (65, 64),      # Asymmetric rectangle
    (128, 128),
    (256, 256),
    (512, 512)
]

# Output folder
downloads_path = str(pathlib.Path.home() / "Downloads")
output_dir = os.path.join(downloads_path, "centered_white_blocks")
os.makedirs(output_dir, exist_ok=True)

# Generate and save images
for (w, h) in box_sizes:
    # Create black canvas
    canvas = np.zeros((canvas_h, canvas_w), dtype=np.uint8)

    # Compute top-left corner of white block to center it
    x_offset = (canvas_w - w) // 2
    y_offset = (canvas_h - h) // 2

    # Paint white rectangle
    canvas[y_offset:y_offset+h, x_offset:x_offset+w] = 255

    # Save image
    filename = f"centered_white_{w}x{h}.jpg"
    filepath = os.path.join(output_dir, filename)
    cv2.imwrite(filepath, canvas)

print(f"Saved {len(box_sizes)} centered white block images to '{output_dir}'.")


Saved 8 centered white block images to 'C:\Users\adity\Downloads\centered_white_blocks'.
